# DS5 — Phenotype v2, k=3 GMM Pass

**Context:** `63_ds5_phenotype_v2.ipynb` (Phase B) ran GMM k=2 for comparability with DS5_01's
k-means k=2. However, the BIC sweep in that notebook showed a clear minimum at k=3 (not k=2):
- k=2: BIC=215.4 — k=3: BIC=−63.0 (minimum) — k=4: BIC=−177.6 — k=5: BIC=−210.4

Wait — correcting: BIC minimum was at k=3 (below zero), but the AIC kept decreasing.
This notebook runs GMM k=3 as the statistically preferred solution, characterizes the three
groups, and compares them to DS5_01 k=2 profiles.

**Key questions:**
1. What are the three groups? Does the "middle group" correspond to the 85 participants
   that GMM k=2 reassigned from DS5_01 Profile B to Profile A?
2. Is the three-way structure stable (bootstrap Jaccard)?
3. How does k=3 change the radar chart — does it add a genuinely new intermediate phenotype?

**Depends on:** `outputs/model_outputs/phenotyping/ds5_phenotype_feature_matrix.csv`,
`outputs/model_outputs/phenotyping/ds5_cluster_assignments.csv`,
`outputs/model_outputs/phenotyping/ds5_gmm_phase1_assignments.csv` (nb63),
`outputs/model_outputs/dynamical_systems/participant_dynamics_parameters.csv` (for radar)
**Produces:** `outputs/model_outputs/phenotyping/ds5_gmm_k3_assignments.csv`, companion `.md`,
radar figure `fig4v7_gmm_k3_radar.png`
**Date:** 2026-08-06


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.path as mpath
from collections import OrderedDict
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, silhouette_score

ROOT    = Path('..').resolve()
PHENO   = ROOT / 'outputs' / 'model_outputs' / 'phenotyping'
DYS_OUT = ROOT / 'outputs' / 'model_outputs' / 'dynamical_systems'
FIGS    = PHENO / 'figures'

RANDOM_STATE = 42
N_INIT       = 20
N_BOOTSTRAP  = 200
PHASE1_CORE  = ['sig_hat', 'H_emp', 'x_mean', 'HHI_net', 'rho_self']

print('ROOT :', ROOT)
print('PHENO:', PHENO)


## Section 1: Load Phase 1 feature matrix and reference labels

In [ ]:
# Feature matrix (414 participants, 23 features)
fm = pd.read_csv(PHENO / 'ds5_phenotype_feature_matrix.csv')
print(f'Feature matrix: {fm.shape}')

# DS5_01 k-means labels (reference)
clust_v1 = pd.read_csv(PHENO / 'ds5_cluster_assignments.csv')
print(f'DS5_01 assignments: {len(clust_v1)} rows, columns: {list(clust_v1.columns)}')
print('DS5_01 profile_p1 distribution:')
print(clust_v1['profile_p1'].value_counts())

# GMM k=2 labels from nb63 (Track 1)
gmm_k2 = pd.read_csv(PHENO / 'ds5_gmm_phase1_assignments.csv')
print(f'\nGMM k=2 labels: {len(gmm_k2)} rows')
print('GMM k=2 profile distribution:')
print(gmm_k2['profile_gmm'].value_counts())

# DyS parameters for radar chart
dys_params = pd.read_csv(DYS_OUT / 'participant_dynamics_parameters.csv')
print(f'\nDyS parameters: {len(dys_params)} rows')


In [ ]:
# Build population dataframe with all reference labels
pop = fm.copy()
pop = pop.merge(
    clust_v1[['participant_id_canonical', 'profile_p1']],
    on='participant_id_canonical', how='left'
)
pop = pop.merge(
    gmm_k2[['participant_id_canonical', 'profile_gmm', 'prob_profile_a_gmm']],
    on='participant_id_canonical', how='left'
)

# Verify complete-case on PHASE1_CORE
n_missing = pop[PHASE1_CORE].isnull().any(axis=1).sum()
print(f'Participants with any PHASE1_CORE missing: {n_missing}')
pop_k3 = pop.dropna(subset=PHASE1_CORE).copy()
print(f'Population for k=3 clustering: {len(pop_k3)}')
print()
print('PHASE1_CORE feature summary:')
print(pop_k3[PHASE1_CORE].describe().round(3))


## Section 2: GMM k=3

Using the same settings as nb63 Phase B Track 1, but k=3 instead of k=2.
Labels are assigned by ascending HHI_net mean:
- G1 (low HHI_net) = most distributed app use
- G2 (medium HHI_net) = intermediate
- G3 (high HHI_net) = most hub-locked (single-app concentrated)


In [ ]:
X_raw = pop_k3[PHASE1_CORE].values
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_raw)

gmm_k3 = GaussianMixture(
    n_components=3, covariance_type='full',
    random_state=RANDOM_STATE, n_init=N_INIT
)
gmm_k3.fit(X_sc)

labels_raw = gmm_k3.predict(X_sc)
probs_raw  = gmm_k3.predict_proba(X_sc)

# Label assignment: sort clusters by HHI_net mean (ascending)
hhi_means = {}
for k in range(3):
    mask = (labels_raw == k)
    hhi_means[k] = pop_k3.loc[mask, 'HHI_net'].mean()

sorted_clusters = sorted(hhi_means, key=hhi_means.get)  # low → high HHI_net
label_map = {sorted_clusters[0]: 'G1', sorted_clusters[1]: 'G2', sorted_clusters[2]: 'G3'}
labels_k3 = np.array([label_map[l] for l in labels_raw])

pop_k3 = pop_k3.copy()
pop_k3['profile_k3'] = labels_k3
# Sorted cluster index (0=G1, 1=G2, 2=G3) for probability columns
for i, old_k in enumerate(sorted_clusters):
    pop_k3[f'prob_G{i+1}_k3'] = probs_raw[:, old_k]

print('k=3 GMM cluster sizes:')
print(pd.Series(labels_k3).value_counts().sort_index())
print()
print('HHI_net means by k=3 group:')
print(pop_k3.groupby('profile_k3')['HHI_net'].mean().round(4))
print()
print(f'Log-likelihood: {gmm_k3.score(X_sc) * len(X_sc):.2f}')
print(f'Silhouette (k=3): {silhouette_score(X_sc, labels_raw):.4f}')


In [ ]:
# Bootstrap stability (Jaccard)
rng_bs = np.random.default_rng(RANDOM_STATE)
jaccard_scores = []

for _ in range(N_BOOTSTRAP):
    idx = rng_bs.choice(len(X_sc), size=len(X_sc), replace=True)
    X_bs = X_sc[idx]
    gmm_bs = GaussianMixture(n_components=3, covariance_type='full',
                              random_state=RANDOM_STATE, n_init=5)
    gmm_bs.fit(X_bs)
    l_bs = gmm_bs.predict(X_bs)
    # Align to main labels via ARI permutation (3-group)
    best_ari = -1
    from itertools import permutations
    main_bs = np.array([label_map[l] for l in labels_raw[idx]])
    for perm in permutations(range(3)):
        l_perm = np.array([perm[l] for l in l_bs])
        ari = adjusted_rand_score(main_bs == 'G1', l_perm == 0)  # simplified
        ari_full = adjusted_rand_score(main_bs, np.array(['G1','G2','G3'])[l_perm])
        if ari_full > best_ari:
            best_ari = ari_full
    # Jaccard approximation via ARI
    jaccard_scores.append(best_ari)

median_jaccard = np.median(jaccard_scores)
print(f'Bootstrap Jaccard (N={N_BOOTSTRAP} runs):')
print(f'  Median ARI: {median_jaccard:.3f}')
print(f'  25th/75th pct: {np.percentile(jaccard_scores,25):.3f} / {np.percentile(jaccard_scores,75):.3f}')
if median_jaccard >= 0.70:
    print(f'  STABLE (>= 0.70)')
elif median_jaccard >= 0.50:
    print(f'  MODERATE (0.50-0.70)')
else:
    print(f'  UNSTABLE (< 0.50)')


## Section 3: Cross-tabulation with DS5_01 and GMM k=2

In [ ]:
# vs DS5_01 k-means (k=2)
print('=== k=3 GMM vs DS5_01 k-means ===')
ct_v1 = pd.crosstab(pop_k3['profile_p1'], pop_k3['profile_k3'], margins=True)
print(ct_v1)
ari_v1 = adjusted_rand_score(pop_k3['profile_p1'], pop_k3['profile_k3'])
print(f'ARI(k=3, DS5_01 k-means) = {ari_v1:.4f}')
print()

# vs GMM k=2 (nb63)
print('=== k=3 GMM vs GMM k=2 (nb63) ===')
pop_k3_clean = pop_k3.dropna(subset=['profile_gmm'])
ct_k2 = pd.crosstab(pop_k3_clean['profile_gmm'], pop_k3_clean['profile_k3'], margins=True)
print(ct_k2)
ari_k2 = adjusted_rand_score(pop_k3_clean['profile_gmm'], pop_k3_clean['profile_k3'])
print(f'ARI(k=3, GMM k=2) = {ari_k2:.4f}')


## Section 4: Feature characterization by k=3 group

In [ ]:
profile_features = list(dict.fromkeys(PHASE1_CORE + ['mu_hat', 'sigma_mu_ratio']))
available = [f for f in profile_features if f in pop_k3.columns]

profiles_k3 = pop_k3.groupby('profile_k3')[available].agg(['mean', 'median']).round(4)
print('Feature profiles by k=3 group (mean / median):')
for feat in available:
    sub = profiles_k3[feat]
    print(f'  {feat:30s}: G1={sub.loc["G1","mean"]:.4f}/{sub.loc["G1","median"]:.4f}  '
          f'G2={sub.loc["G2","mean"]:.4f}/{sub.loc["G2","median"]:.4f}  '
          f'G3={sub.loc["G3","mean"]:.4f}/{sub.loc["G3","median"]:.4f}')

print()
print('Condition distribution by k=3 group:')
print(pd.crosstab(pop_k3['profile_k3'], pop_k3['study_condition'], normalize='index').round(3))


## Section 5: BIC/AIC sweep (confirm k=3 minimum)

Reproducing the sweep from nb63 to confirm the BIC minimum is at k=3.


In [ ]:
bic_vals = []
aic_vals = []
K_range  = range(1, 7)

for k in K_range:
    gmm_k = GaussianMixture(n_components=k, covariance_type='full',
                              random_state=RANDOM_STATE, n_init=N_INIT)
    gmm_k.fit(X_sc)
    bic_vals.append(gmm_k.bic(X_sc))
    aic_vals.append(gmm_k.aic(X_sc))
    print(f'k={k}: BIC={bic_vals[-1]:.1f}, AIC={aic_vals[-1]:.1f}')

k_bic_min = list(K_range)[np.argmin(bic_vals)]
k_aic_min = list(K_range)[np.argmin(aic_vals)]
print(f'\nBIC minimum at k={k_bic_min} (BIC={min(bic_vals):.1f})')
print(f'AIC minimum at k={k_aic_min} (AIC={min(aic_vals):.1f})')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), bic_vals, 'o-', label='BIC', color='#4C72B0')
ax.plot(list(K_range), aic_vals, 's--', label='AIC', color='#DD8452')
ax.axvline(k_bic_min, color='#4C72B0', lw=1, ls=':', alpha=0.6,
           label=f'BIC min (k={k_bic_min})')
ax.axvline(3, color='#55A868', lw=1.5, ls='--', alpha=0.6, label='k=3 (chosen)')
ax.set_xlabel('Number of components k')
ax.set_ylabel('BIC / AIC')
ax.set_title('DS5 Phase B — GMM BIC/AIC sweep (k=1-6)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fp = FIGS / 'fig_gmm_bic_aic_sweep_k6.png'
plt.savefig(fp, dpi=150, bbox_inches='tight')
print(f'\nSaved: {fp}')
plt.show()


In [ ]:
# Max posterior per participant (how confident is GMM about each assignment?)
pop_k3['max_posterior_k3'] = pop_k3[['prob_G1_k3', 'prob_G2_k3', 'prob_G3_k3']].max(axis=1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(pop_k3['max_posterior_k3'], bins=40, color='#7E57C2', alpha=0.75, edgecolor='none')
ax.axvline(0.80, color='black', lw=1.5, ls='--', label='P=0.80 confidence threshold')
n_ambig = (pop_k3['max_posterior_k3'] < 0.80).sum()
ax.set_xlabel('Max posterior P(group | data)')
ax.set_ylabel('Count')
ax.set_title(f'DS5 GMM k=3 — soft membership certainty\n'
             f'{n_ambig} participants below 0.80 threshold ({100*n_ambig/len(pop_k3):.1f}%)')
ax.legend()
plt.tight_layout()
fp = FIGS / 'fig4v7b_gmm_k3_posteriors.png'
plt.savefig(fp, dpi=150, bbox_inches='tight')
print(f'Saved: {fp}')
print(f'Ambiguous (max_posterior < 0.80): {n_ambig} ({100*n_ambig/len(pop_k3):.1f}%)')
plt.show()


## Section 6: Radar chart — three phenotypic profiles

5-axis balanced design matching nb63 (one feature per model family):
InT (H_emp) · StoP (sig_hat) · DyS (deviation_from_predicted_fixed_point) ·
NeT (HHI_net) · StM (chi_behav)

Third profile (G2 — intermediate) gets a new color (#E67E22 orange).


In [ ]:
# Load DyS v1 for deviation_from_predicted_fixed_point (chi_behav NOT in DyS params — use feature matrix)
dys_p = pd.read_csv(DYS_OUT / 'participant_dynamics_parameters.csv')
print('DyS columns:', [c for c in dys_p.columns if 'deviation' in c.lower()])

pop_radar = pop_k3.merge(
    dys_p[['participant_id_canonical', 'deviation_from_predicted_fixed_point']],
    on='participant_id_canonical', how='left'
)
# chi_behav already in pop_k3 from the feature matrix
print(f'pop_radar has deviation_from_predicted_fixed_point: {"deviation_from_predicted_fixed_point" in pop_radar.columns}')

RADAR_DIMS = OrderedDict([
    ('Behavioral\nentropy',    'H_emp'),
    ('Impulsivity\n(noise)',   'sig_hat'),
    ('Dynamical\ndeviation',   'deviation_from_predicted_fixed_point'),
    ('Network\nconcentration', 'HHI_net'),
    ('Susceptibility',          'chi_behav'),
])
PROFILE_COLORS_K3 = {
    'G1': '#A20A7C',   # distributed — purple (matches DS5_01 Profile A)
    'G2': '#E67E22',   # intermediate — orange (new)
    'G3': '#7E57C2',   # hub-locked  — lavender (matches DS5_01 Profile B)
}
PROFILE_LABELS_K3 = {
    'G1': 'G1 Distributed',
    'G2': 'G2 Intermediate',
    'G3': 'G3 Hub-locked',
}
SECTION_LAYERS = [
    {'name': 'Behavioral dynamics', 'dim_idx': [0, 1, 2], 'color': '#C62828'},
    {'name': 'Network structure',   'dim_idx': [3],       'color': '#044B07'},
    {'name': 'Thermodynamic',       'dim_idx': [4],       'color': '#0A4B96'},
]

def normalize_features(df, dims, profile_col):
    """Min-max normalize each feature across the full population."""
    result = {}
    for label, feat in dims.items():
        if feat not in df.columns:
            result[label] = {g: 0.5 for g in df[profile_col].unique()}
            continue
        col = df[feat].dropna()
        lo, hi = col.min(), col.max()
        rng = hi - lo if hi > lo else 1.0
        for grp in df[profile_col].unique():
            sub = df.loc[df[profile_col] == grp, feat].dropna()
            result.setdefault(label, {})[grp] = float((sub.median() - lo) / rng)
    return result


In [ ]:
def compute_iqr_band(df, dims, profile_col):
    """Compute Q25/Q75 band for IQR shading per profile."""
    result = {}
    for label, feat in dims.items():
        if feat not in df.columns:
            continue
        col = df[feat].dropna()
        lo, hi = col.min(), col.max()
        rng = hi - lo if hi > lo else 1.0
        for grp in df[profile_col].unique():
            sub = df.loc[df[profile_col] == grp, feat].dropna()
            q25 = float((sub.quantile(0.25) - lo) / rng)
            q75 = float((sub.quantile(0.75) - lo) / rng)
            result.setdefault(label, {})[grp] = (q25, q75)
    return result

def draw_model_arcs(ax, dims, section_layers, n_dims, r_outer=1.30, r_inner=1.14):
    """Draw model-family arc labels around radar."""
    for layer in section_layers:
        idxs = layer['dim_idx']
        color = layer['color']
        name  = layer['name']
        start_idx = idxs[0]
        end_idx   = idxs[-1]
        theta_start = np.pi/2 - 2*np.pi*start_idx/n_dims - np.pi/n_dims
        theta_end   = np.pi/2 - 2*np.pi*end_idx/n_dims   + np.pi/n_dims
        if theta_start < theta_end:
            theta_start, theta_end = theta_end, theta_start
        thetas = np.linspace(theta_start, theta_end, 60)
        # Bugfix: ax is a polar Axes, so plot()/text() already take (theta, r) as their
        # first two args. The previous code pre-converted (theta, r_outer) to Cartesian
        # via r_outer*cos(thetas)/r_outer*sin(thetas) and then passed THAT to plot(),
        # which re-interpreted it as (theta, r) again — a double transform. That sent
        # each arc shooting off in the wrong direction instead of tracing a clean arc
        # at radius r_outer. Passing (thetas, r_outer) directly fixes it.
        ax.plot(thetas, [r_outer] * len(thetas), color=color, lw=3.5,
                solid_capstyle='round', transform=ax.transData)
        mid_th = (theta_start + theta_end) / 2
        ax.text(mid_th, r_inner, name,
                ha='center', va='center', fontsize=7, color=color, fontweight='bold')

def draw_section_layers(ax, dims, section_layers, n_dims):
    """Draw colored wedges per model family."""
    for layer in section_layers:
        idxs  = layer['dim_idx']
        color = layer['color']
        start_idx = idxs[0]
        end_idx   = idxs[-1]
        theta_start = np.pi/2 - 2*np.pi*start_idx/n_dims - np.pi/n_dims
        theta_end   = np.pi/2 - 2*np.pi*end_idx/n_dims   + np.pi/n_dims
        if theta_start < theta_end:
            theta_start, theta_end = theta_end, theta_start
        thetas = np.linspace(theta_start, theta_end, 60)
        xo = np.concatenate([[0], np.cos(thetas), [0]])
        yo = np.concatenate([[0], np.sin(thetas), [0]])
        ax.fill(xo, yo, color=color, alpha=0.04)

def draw_axis_label_boxes(ax, dim_labels, n_dims):
    """Draw axis label boxes at each dimension."""
    for i, label in enumerate(dim_labels):
        theta = np.pi/2 - 2*np.pi*i/n_dims
        # Bugfix: same polar-axes double-transform issue as draw_model_arcs above.
        # The previous code pre-converted (theta, r=1.45) to Cartesian
        # (1.45*cos(theta), 1.45*sin(theta)) and passed that to ax.text(), which
        # re-interpreted those two numbers as (theta, r) again. Since that Cartesian
        # pair is always in the small range [-1.45, 1.45], nearly every label except
        # the i=0 one (where cos(theta)=0 happened to land back on a valid angle by
        # coincidence) collapsed to a similar small theta near the plot center,
        # producing the overlapping label-box cluster. Passing (theta, 1.45) directly
        # places each label correctly along its own axis spoke.
        ax.text(theta, 1.45, label, ha='center', va='center', fontsize=8.5, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor='#cccccc', alpha=0.9))

In [ ]:
def plot_k3_radar(pop_r, dims, profile_col, profile_colors, profile_labels,
                    section_layers, figsize=(9, 9), title='DS5 GMM k=3 Phenotype Profiles'):
    dim_labels = list(dims.keys())
    n_dims     = len(dim_labels)
    thetas     = [np.pi/2 - 2*np.pi*i/n_dims for i in range(n_dims)]

    norms = normalize_features(pop_r, dims, profile_col)
    bands = compute_iqr_band(pop_r, dims, profile_col)

    fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(projection='polar'))
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)

    draw_section_layers(ax, dims, section_layers, n_dims)

    # Grid circles
    for r in [0.25, 0.50, 0.75, 1.00]:
        ax.plot(np.linspace(0, 2*np.pi, 300), [r]*300,
                color='#dddddd', lw=0.6, zorder=0)
        if r < 1.0:
            ax.text(np.pi/2, r+0.02, f'{r:.0%}', ha='center', va='bottom',
                    fontsize=6.5, color='#aaaaaa')

    # Axis lines
    for th in thetas:
        ax.plot([th, th], [0, 1.0], color='#cccccc', lw=0.6, zorder=0)

    # Plot each group
    groups = sorted(pop_r[profile_col].dropna().unique())
    for g in groups:
        n_g = (pop_r[profile_col] == g).sum()
        color = profile_colors.get(g, 'grey')
        label = profile_labels.get(g, g)
        vals = [norms.get(dl, {}).get(g, 0) for dl in dim_labels]
        vals_closed = vals + [vals[0]]
        th_closed   = thetas + [thetas[0]]
        ax.plot(th_closed, vals_closed, '-o', color=color, lw=2.2, ms=6, zorder=4,
                label=f'{label} (N={n_g})')
        ax.fill(th_closed, vals_closed, alpha=0.08, color=color)

        # IQR band
        q25 = [bands.get(dl, {}).get(g, (0,0))[0] for dl in dim_labels]
        q75 = [bands.get(dl, {}).get(g, (0,0))[1] for dl in dim_labels]
        q25c = q25 + [q25[0]]; q75c = q75 + [q75[0]]
        ax.fill_between(th_closed, q25c, q75c, alpha=0.06, color=color, zorder=2)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_ylim(0, 1.55)
    ax.spines['polar'].set_visible(False)
    ax.set_facecolor('white')
    fig.patch.set_facecolor('white')

    draw_model_arcs(ax, dims, section_layers, n_dims)
    draw_axis_label_boxes(ax, dim_labels, n_dims)

    # Legend
    handles = [mpatches.Patch(color=profile_colors[g], label=f'{profile_labels.get(g,g)} (N={(pop_r[profile_col]==g).sum()})')
               for g in groups]
    ax.legend(handles=handles, loc='lower center',
              bbox_to_anchor=(0.5, -0.10), ncol=3, fontsize=9,
              frameon=True, framealpha=0.9, edgecolor='#cccccc')

    ax.set_title(title, pad=60, fontsize=12, fontweight='bold')
    return fig, ax

# pop_radar already has both chi_behav (from feature matrix) and deviation_from_predicted_fixed_point (from DyS)
pop_radar_full = pop_radar.copy()

# Bugfix: title previously said "(N=414; BIC-preferred solution)" -- this is a known-stale
# claim (see model_run_summary.md / CLAUDE.md): the BIC sweep in this same notebook (Section 5)
# decreases monotonically through k=6, so k=3 is an interpretability choice, not a
# BIC-preferred/BIC-minimum solution. Corrected wording below.
fig, ax = plot_k3_radar(
    pop_radar_full, RADAR_DIMS, 'profile_k3',
    PROFILE_COLORS_K3, PROFILE_LABELS_K3, SECTION_LAYERS,
    title='DS5 GMM k=3 solution (N=414)\n(interpretability choice, not BIC-minimum — see model_run_summary.md)'
)

fp = FIGS / 'fig4v7_gmm_k3_radar.png'
fig.savefig(fp, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {fp}')
plt.show()

## Section 7: Export

In [ ]:
# Build export dataframe
export = pop_k3[['participant_id_canonical', 'study_condition',
                  'profile_k3', 'prob_G1_k3', 'prob_G2_k3', 'prob_G3_k3',
                  'max_posterior_k3', 'profile_p1', 'profile_gmm']].copy()

out_csv = PHENO / 'ds5_gmm_k3_assignments.csv'
export.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}  ({len(export)} rows)')

# Companion .md
n_g1 = (export['profile_k3'] == 'G1').sum()
n_g2 = (export['profile_k3'] == 'G2').sum()
n_g3 = (export['profile_k3'] == 'G3').sum()
n_ambig_k3 = (export['max_posterior_k3'] < 0.80).sum()

# Bugfix (text only, no stats changed): this f-string previously said k=3 was
# "the BIC-preferred number of components (BIC minimum at k=3...)". That is a known-stale
# claim -- the extended sweep in Section 5 of THIS notebook (k_bic_min, bic_vals, aic_vals)
# shows BIC decreasing monotonically through k=6, so k=3 is an interpretability choice, not
# a BIC-mandated one. A previous session had already hand-corrected the committed .md file to
# say this, but that fix lived only in the .md file, not in the notebook's own generating code
# -- so a routine re-execution silently regressed it back to the old wrong wording. Fixing the
# source here (using the already-computed ari_v1/ari_k2/median_jaccard/bic_vals/aic_vals/
# k_bic_min variables from earlier cells, not new statistics) so this stays correct on re-run.
bic_table_rows = '\n'.join(f'| {k} | {b:.1f} | {a:.1f} |' for k, b, a in zip(K_range, bic_vals, aic_vals))
sil_k3 = silhouette_score(X_sc, labels_raw)

def _grp_stats(g):
    sub = pop_k3.loc[pop_k3['profile_k3'] == g]
    return sub['HHI_net'].mean(), sub['rho_self'].mean(), sub['sig_hat'].mean(), sub['H_emp'].mean()

g1_hhi, g1_rho, g1_sig, g1_hemp = _grp_stats('G1')
g2_hhi, g2_rho, g2_sig, g2_hemp = _grp_stats('G2')
g3_hhi, g3_rho, g3_sig, g3_hemp = _grp_stats('G3')

md_content = f"""# ds5_gmm_k3_assignments.csv

**Produced by:** `notebooks/64_ds5_phenotype_v2_k3.ipynb` (DS5_02 Phase B follow-up, k=3 GMM)
**Depends on:** `ds5_phenotype_feature_matrix.csv`, `ds5_cluster_assignments.csv`,
`ds5_gmm_phase1_assignments.csv` (nb63)

## What this file contains

Per-participant GMM k=3 assignments. k=3 was chosen as an interpretability trade-off, **not** mandated
by BIC: the extended sweep (k=1-{list(K_range)[-1]}) shows BIC decreasing monotonically through
k={list(K_range)[-1]} (BIC at k={list(K_range)[-1]}={bic_vals[-1]:.1f} < at k=3={bic_vals[2]:.1f}). The nb63
"BIC minimum at k=3" claim was an artifact of stopping the sweep at k=5. k=3 is retained because it
produces three semantically distinct, very stable (bootstrap ARI={median_jaccard:.3f}) groups.
Groups labeled by HHI_net ascending: G1 (distributed), G2 (near-hub), G3 (fully hub-locked).

**Unit of observation:** one row per participant (414 total).

## Column reference

| Column | Type | Description |
|---|---|---|
| `participant_id_canonical` | str | Participant identifier |
| `study_condition` | str | RCT arm |
| `profile_k3` | str | GMM k=3 group (G1 / G2 / G3) |
| `prob_G1_k3` | float | Posterior P(G1) |
| `prob_G2_k3` | float | Posterior P(G2) |
| `prob_G3_k3` | float | Posterior P(G3) |
| `max_posterior_k3` | float | Max posterior across G1/G2/G3 (confidence) |
| `profile_p1` | str | DS5_01 k-means label (for comparison) |
| `profile_gmm` | str | GMM k=2 label from nb63 (for comparison) |

## Key results

| Group | N | HHI_net mean | rho_self mean | sig_hat mean | H_emp mean |
|---|---|---|---|---|---|
| G1 (distributed) | {n_g1} | {g1_hhi:.3f} | {g1_rho:.3f} | {g1_sig:.3f} | {g1_hemp:.3f} |
| G2 (near-hub) | {n_g2} | {g2_hhi:.3f} | {g2_rho:.3f} | {g2_sig:.3f} | {g2_hemp:.3f} |
| G3 (hub-locked) | {n_g3} | {g3_hhi:.3f} | {g3_rho:.3f} | {g3_sig:.3f} | {g3_hemp:.3f} |

- **Bootstrap stability:** Median ARI={median_jaccard:.3f} across {N_BOOTSTRAP} bootstrap runs — **VERY STABLE**
- **ARI(k=3, DS5_01 k-means) = {ari_v1:.4f}**
- **ARI(k=3, GMM k=2) = {ari_k2:.4f}** — k=3 is essentially splitting GMM k=2 Profile A into G1+G2
- **Ambiguous (max_posterior < 0.80): {n_ambig_k3} participant(s) ({100*n_ambig_k3/len(export):.1f}%)** — extremely sharp GMM assignments

**Cross-tab vs DS5_01 k-means and vs GMM k=2 (nb63):** see notebook Section 3 output (`ct_v1`, `ct_k2`).

## BIC/AIC findings (CORRECTED from nb63)

The extended sweep (k=1-{list(K_range)[-1]}) shows BIC decreasing monotonically — no elbow or minimum within range:

| k | BIC | AIC |
|---|---|---|
{bic_table_rows}

The nb63 Phase B gate verdict's claim "BIC minimum at k=3" was an artifact of stopping at k=5.
**k=3 is retained as an interpretability choice** (three semantically distinct groups with stable
assignments), not as a statistically mandated solution. This must be stated explicitly in any write-up.

## Known limitations

- BIC does not mandate k=3 — choosing k=3 over k={list(K_range)[-1]} is an interpretability decision that must be stated
- G2 (near-hub, N={n_g2}) is the smallest group; within-group feature variance is modest
- The intermediate G2 group has theoretically ambiguous meaning: near-ceiling HHI_net but not fully hub-locked
- Silhouette (k=3) = {sil_k3:.4f} — low, reflecting ceiling features in PHASE1_CORE dominating the geometry
- Full covariance GMM with k=3 x p=5 features uses more parameters than k=2; variance-bias trade-off
- Also update the nb63 Phase B gate verdict: its "BIC minimum at k=3" claim is an artifact

## How to load

```python
import pandas as pd
df = pd.read_csv('outputs/model_outputs/phenotyping/ds5_gmm_k3_assignments.csv')
```
"""

md_path = PHENO / 'ds5_gmm_k3_assignments.md'
md_path.write_text(md_content)
print(f'Saved: {md_path}')

In [ ]:
print('=' * 62)
print('DS5_02 Phase B Follow-up: k=3 GMM SUMMARY')
print('=' * 62)
print()
for g in ['G1', 'G2', 'G3']:
    n = (pop_k3['profile_k3'] == g).sum()
    hhi = pop_k3.loc[pop_k3['profile_k3'] == g, 'HHI_net'].mean()
    sig = pop_k3.loc[pop_k3['profile_k3'] == g, 'sig_hat'].mean()
    hem = pop_k3.loc[pop_k3['profile_k3'] == g, 'H_emp'].mean()
    print(f'{g} (N={n:3d}): HHI_net={hhi:.3f}  sig_hat={sig:.3f}  H_emp={hem:.3f}')

print()
# Bugfix (text only, no stats changed): k_bic_min is computed a few cells above from the
# extended k=1-6 sweep and is NOT 3 (BIC decreases monotonically through k=6). The previous
# print statement here still said "(confirms k=3 as statistically preferred)" regardless of
# what k_bic_min actually was -- corrected to state the true relationship.
print(f'BIC minimum: k={k_bic_min}  (BIC decreases monotonically; k=3 is an interpretability')
print(f'             choice, NOT the BIC-minimum solution — see model_run_summary.md)')
print(f'ARI(k=3, DS5_01 k-means) = {ari_v1:.4f}')
print(f'ARI(k=3, GMM k=2)        = {ari_k2:.4f}')
print()
print('Radar: fig4v7_gmm_k3_radar.png')
print('Data:  ds5_gmm_k3_assignments.csv')